# Exercise 03 — Environment Config

Every AI project has secrets: API keys for OpenAI, Anthropic, Pinecone, whatever you're using. The first rule is simple: never hardcode them. Not in your script, not in a comment, not ever — because secrets in code end up in git, and git is forever.

The standard approach is a `.env` file that you load at runtime. This exercise shows you how to do that locally and in Colab.

**What you'll practice:**
- Loading secrets from a `.env` file with `python-dotenv`
- Using Colab's built-in secrets manager (the safe way in the cloud)
- Building a typed config object so your app always knows what it needs

In [ ]:
!pip install python-dotenv pydantic -q

## Part 1 — Loading from a .env file

When running locally, you keep your secrets in a `.env` file at the root of the project and add it to `.gitignore`. `python-dotenv` reads it and puts the values into `os.environ`.

In [ ]:
# Create a sample .env file for this exercise
# In a real project this file already exists and is in .gitignore

env_content = """OPENAI_API_KEY=sk-fake-key-for-exercise
APP_ENV=development
MAX_TOKENS=2048
"""

with open(".env", "w") as f:
    f.write(env_content)

print(".env file created")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env and populates os.environ

api_key = os.getenv("OPENAI_API_KEY")
app_env = os.getenv("APP_ENV", "production")  # second arg is the default
max_tokens = int(os.getenv("MAX_TOKENS", "1024"))

print(f"Environment: {app_env}")
print(f"Max tokens: {max_tokens}")
print(f"API key loaded: {'yes' if api_key else 'no'}")

## Part 2 — Colab secrets

In Colab, don't use `.env` files. Use the built-in secrets manager instead: click the key icon in the left sidebar, add your secret, then access it with `userdata.get()`.

In [ ]:
# Run this only in Google Colab
# Add your key in the sidebar first: key icon -> "Add new secret"

try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
    print("Running in Colab, key loaded from secrets.")
except ImportError:
    print("Not in Colab. Using .env file instead.")

## Part 3 — Typed config object

Reading from `os.environ` everywhere gets messy. A better pattern is to load everything once into a typed object and pass it around. Pydantic makes this clean.

In [ ]:
from pydantic import BaseModel


class AppConfig(BaseModel):
    openai_api_key: str
    app_env: str = "production"
    max_tokens: int = 1024

    @classmethod
    def from_env(cls) -> "AppConfig":
        return cls(
            openai_api_key=os.environ["OPENAI_API_KEY"],
            app_env=os.getenv("APP_ENV", "production"),
            max_tokens=int(os.getenv("MAX_TOKENS", "1024")),
        )


config = AppConfig.from_env()
print(config.model_dump(exclude={"openai_api_key"}))

## Check your understanding

- What happens if `OPENAI_API_KEY` is missing from the environment when you call `AppConfig.from_env()`? Try it.
- Why is `os.environ["KEY"]` different from `os.getenv("KEY")`? When do you want each one?
- Add a `model_name: str` field to `AppConfig` with a sensible default.